In [ ]:
install.packages("BiocManager")

BiocManager::install("ensembldb")
BiocManager::install("AnnotationHub")
BiocManager::install("GenomicRanges")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Installing package(s) 'BiocVersion', 'ensembldb'

also installing the dependencies ‘matrixStats’, ‘abind’, ‘SparseArray’, ‘formatR’, ‘MatrixGenerics’, ‘S4Arrays’, ‘DelayedArray’, ‘lambda.r’, ‘futile.options’, ‘png’, ‘SummarizedExperiment’, ‘cigarillo’, ‘RCurl’, ‘rjson’, ‘futile.logger’, ‘snow’, ‘BH’, ‘XVector’, ‘lazyeval’, ‘UCSC.utils’, ‘KEGGREST’, ‘XML’, ‘GenomicAlignments’, ‘BiocIO’, ‘restfulr’, ‘bitops’, ‘BiocParallel’, ‘Rhtslib’, ‘BiocGenerics’, ‘GenomicRanges’, ‘GenomicFeatures’, ‘AnnotationFilter’, ‘RSQLite’, ‘Biobase’, ‘Seqinfo’, ‘GenomeInfoDb’, ‘AnnotationDbi’, ‘rtracklayer’, ‘S4Vectors’, ‘Rsamtools’, ‘IRanges’, ‘ProtGenerics’, ‘Biostrin

In [ ]:
library(ensembldb)
library(AnnotationHub)
library(GenomicRanges)

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following object is masked from ‘pac

In [ ]:
ah <- AnnotationHub()

edb <- ah[["AH119325"]]
gene_symbol <- "PLN"

tx_df <- transcripts(
  edb,
  filter = GeneNameFilter(gene_symbol),
  return.type = "DataFrame"
)
tx_df

downloading 1 resources

retrieving 1 resource



loading from cache



DataFrame with 2 rows and 14 columns
            tx_id     tx_biotype tx_seq_start tx_seq_end tx_cds_seq_start
      <character>    <character>    <integer>  <integer>        <integer>
1       LRG_390t1       LRG_gene    118548279  118560424        118558922
2 ENST00000357525 protein_coding    118548296  118561716        118558922
  tx_cds_seq_end         gene_id tx_support_level     tx_id_version gc_content
       <integer>     <character>        <integer>       <character>  <numeric>
1      118559080         LRG_390               NA       LRG_390t1.1    33.0805
2      118559080 ENSG00000198523                1 ENST00000357525.6    31.7497
  tx_external_name tx_is_canonical         tx_name   gene_name
       <character>       <integer>     <character> <character>
1               NA               1       LRG_390t1         PLN
2          PLN-201               1 ENST00000357525         PLN

In [ ]:
write.table(
  as.data.frame(tx_df),
  file = "transcripts.tsv",
  sep = "\t",
  row.names = FALSE,
  quote = FALSE
)
pr_df <- proteins(
  edb,
  filter = GeneNameFilter(gene_symbol),
  return.type = "DataFrame"
)
pr_df

DataFrame with 2 rows and 4 columns
            tx_id      protein_id       protein_sequence   gene_name
      <character>     <character>            <character> <character>
1 ENST00000357525 ENSP00000350132 MEKVQYLTRSAIRRASTIEM..         PLN
2       LRG_390t1       LRG_390p1 MEKVQYLTRSAIRRASTIEM..         PLN

In [ ]:
write.table(
  as.data.frame(pr_df),
  file = "proteins.tsv",
  sep = "\t",
  row.names = FALSE,
  quote = FALSE
)
dir.create("protein_maps", showWarnings = FALSE)

for (i in seq_len(nrow(pr_df))) {

  pid  <- pr_df$protein_id[i]

  pseq <- pr_df$protein_sequence[i]

  if (is.na(pid) || is.na(pseq) || nchar(pseq) == 0)
    next

  rng <- IRanges(
    start = 1,
    end = nchar(pseq),
    names = pid
  )

  map <- proteinToGenome(rng, edb)[[1]]

  write.table(
    as.data.frame(map),
    file = file.path(
      "protein_maps",
      paste0(pid, "_map.tsv")
    ),
    sep = "\t",
    row.names = FALSE,
    quote = FALSE
  )
}
getwd()

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK



[1] "/content"